In [1]:
%%capture
%pip install dspy-ai -U transformers chromadb accelerate sentence-transformers bitsandbytes peft rich

In [2]:
import gc
import dspy
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
from torch import bfloat16
from rich import print

In [3]:
access_token = ""
model_name = "google/gemma-7b"
llm = dspy.HFModel(model=model_name, hf_device_map='auto', token=access_token, model_kwargs= {'temperature': 0.0, 'do_sample': False})
llm.model=None
gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)
llm.model=AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
dspy.settings.configure(lm=llm)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_u

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
class GenerateAnswer(dspy.Signature):
    """Answer questions with short factoid answers."""

    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="often between 1 and 5 words")

In [5]:
class QUESTIONANSWER(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens = 300)

    def forward(self, question,context):
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [6]:
uncompiled_fs = QUESTIONANSWER()

In [7]:
example1 = """
6 hours\nAssociate Actuary - SPA Rx\nCincinnati, OH 45217\n**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a remote nationwide position_ The Associate Actuary, Pricing establishes market level financial metrics to align with segment profitability goals, analyzes market level results and projections and develops recommended pricing actions to address gaps to targeted metrics. Leverages market level projections and experience data tools to research root cause and capture insights. Researches and understands competitors in marketplace and collaborates with sales and other partners supporting the markets to develop strategies for profitable membership growth. Accountable for actuarial certifications on rate filings, including attesting to compliance with state and federal rating and benefit regulations. Begins to influence department's strategy. Makes decisions on moderately complex to complex issues regarding technical approach for project components, and work is performed without direction. Exercises considerable latitude in determining objectives and approaches to assignments. **Required Qualifications** + Bachelor's Degree + Associate of Society of Actuaries (ASA) designation + Meets eligibility requirements for Humana's Actuarial Professional Development Program (APDP) + MAAA + Strong communication + Must be passionate about contributing to an organization focused on continuously improving consumer experiences **Our Hiring Process** As part of our hiring process for this opportunity, we may contact you via text message and email to gather more information using a software platform called Modern Hire. Modern Hire Text, Scheduling and Video technologies allow you to interact with us at the time and location most convenient for you. If you are selected to move forward from your application prescreen, you may receive correspondence inviting you to participate in a pre-recorded Voice, Text Messaging and/or Video interview. Your recorded interview will be reviewed and you will subsequently be informed if you will be moving forward to next round of interviews If you have additional questions regarding this role posting and are an Internal Candidate, please send them to the Ask A Recruiter persona by visiting go/Buzz and searching Ask A Recruiter! Please be sure to provide the requisition number so we may be able to research your request quicker. **Alert:** Humana values personal identity protection. Please be aware that applicants selected for leader review may be asked to provide a social security number, if it is not already on file. When required, an email will be sent from Humana@myworkday.com with instructions to add the information into the application at Humana's secure website. **_Humana is more than an equal opportunity employer, Humana's dedication to promoting diversity, multiculturalism, and inclusion is at the heart of what we do in all of our Humana roles. Diversity is more than a commitment to us, it is the foundation of what we do. We are fully focused on diversity of race, gender, sexual orientation, religion, ethnicity, national origin and all of the other fascinating characteristics that make us each uniquely wonderful._** \#LI-Remote **Scheduled Weekly Hours** 40 Humana complies with all applicable federal civil rights laws and does not discriminate on the basis of race, color, national origin, age, disability, sex, sexual orientation, gender identity or religion. We also provide free language interpreter services. See our https://www.humana.com/legal/accessibility-resources?source=Humana_Website.
"""

In [8]:
print(example1)

6 hours
Associate Actuary - SPA Rx
Cincinnati, OH 45217
**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, 
filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports 
implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new 
product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The 
Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of 
situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a remote 
nationwide position_ The Associate Actuary, Pricing establishes market level financial metrics to align with 
segment profitability goals, analyzes market level results and projections and develops recommended pricing actions
to address gaps to targeted metrics. Leverages market level projections and experience data tools to research root 
cause and capture insights. Researches and understands competitors in marketplace and collaborates with sales and 
other partners supporting the markets to develop strategies for profitable membership growth. Accountable for 
actuarial certifications on rate filings, including attesting to compliance with state and federal rating and 
benefit regulations. Begins to influence department's strategy. Makes decisions on moderately complex to complex 
issues regarding technical approach for project components, and work is performed without direction. Exercises 
considerable latitude in determining objectives and approaches to assignments. **Required Qualifications** + 
Bachelor's Degree + Associate of Society of Actuaries (ASA) designation + Meets eligibility requirements for 
Humana's Actuarial Professional Development Program (APDP) + MAAA + Strong communication + Must be passionate about
contributing to an organization focused on continuously improving consumer experiences **Our Hiring Process** As 
part of our hiring process for this opportunity, we may contact you via text message and email to gather more 
information using a software platform called Modern Hire. Modern Hire Text, Scheduling and Video technologies allow
you to interact with us at the time and location most convenient for you. If you are selected to move forward from 
your application prescreen, you may receive correspondence inviting you to participate in a pre-recorded Voice, 
Text Messaging and/or Video interview. Your recorded interview will be reviewed and you will subsequently be 
informed if you will be moving forward to next round of interviews If you have additional questions regarding this 
role posting and are an Internal Candidate, please send them to the Ask A Recruiter persona by visiting go/Buzz and
searching Ask A Recruiter! Please be sure to provide the requisition number so we may be able to research your 
request quicker. **Alert:** Humana values personal identity protection. Please be aware that applicants selected 
for leader review may be asked to provide a social security number, if it is not already on file. When required, an
email will be sent from Humana@myworkday.com with instructions to add the information into the application at 
Humana's secure website. **_Humana is more than an equal opportunity employer, Humana's dedication to promoting 
diversity, multiculturalism, and inclusion is at the heart of what we do in all of our Humana roles. Diversity is 
more than a commitment to us, it is the foundation of what we do. We are fully focused on diversity of race, 
gender, sexual orientation, religion, ethnicity, national origin and all of the other fascinating characteristics 
that make us each uniquely wonderful._** \#LI-Remote **Scheduled Weekly Hours** 40 Humana complies with all 
applicable federal civil rights laws and does not discriminate on the basis of race, color, national origin, age, 
disa

In [9]:
pred = uncompiled_fs("Where is this job located and is it remote, hybrid or onsite?",example1)
torch.cuda.empty_cache()
gc.collect()

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ValueError: `temperature` (=0.0) has to be a strictly positive float, otherwise your next token scores will be invalid. If you're looking for greedy decoding strategies, set `do_sample=False`.

In [ ]:
pred.answer

In [ ]:
llm.inspect_history()

In [ ]:
pred = uncompiled_fs("What are the skills and education needed in this job?",example1)
torch.cuda.empty_cache()
gc.collect()

In [ ]:
pred.answer

In [ ]:
llm.inspect_history()